# EuroSAT LULC Benchmark: Training, Evaluation & Hugging Face Auto-Sync

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Krishna2805/Eurosat-lulc-benchmark/blob/main/EuroSAT_Benchmark_Training.ipynb)

This notebook trains and benchmarks three computer vision architectures on the **EuroSAT RGB** dataset:
1. **Custom CNN (Swish Activation)**: Baseline CNN trained from scratch (~1.2M parameters)
2. **VGG16 Transfer Learning**: Classical transfer learning baseline (~14.7M parameters)
3. **ResNet50 Transfer Learning**: Deep residual transfer learning (~24.1M parameters)

### Instructions for Google Colab:
- Set runtime to GPU: **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**
- Set your Hugging Face write token in **Secrets** (key: `HF_TOKEN`) or enter it when prompted to auto-sync trained `.h5` weights to `Krishna2805/eurosat-benchmarks`.

In [ ]:
# Step 1: Environment Verification & GPU Detection
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"SUCCESS: GPU detected -> {gpus}")
else:
    print("WARNING: No GPU detected. Running on CPU. Switch to T4 GPU via Runtime -> Change runtime type.")

try:
    import torch
    print("PyTorch CUDA Available:", torch.cuda.is_available())
except ImportError:
    pass

In [ ]:
# Step 2: Hugging Face Configuration
!pip install -q huggingface_hub
from huggingface_hub import HfApi

HF_REPO_ID = "Krishna2805/eurosat-benchmarks"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Check Colab secrets if available
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = input("Enter Hugging Face Write Token (or press Enter to skip auto-upload): ").strip()

if HF_TOKEN:
    print(f"HF_TOKEN loaded. Target repository: {HF_REPO_ID}")
else:
    print("HF_TOKEN not provided. Models will be saved locally only.")

In [ ]:
# Step 3: Dataset Ingestion & Preprocessing
print("Loading EuroSAT RGB dataset from TensorFlow Datasets...")
(ds_train_raw, ds_val_raw, ds_test_raw), ds_info = tfds.load(
    'eurosat/rgb',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    as_supervised=True,
    with_info=True
)

class_names = ds_info.features['label'].names
num_classes = len(class_names)
print(f"Total Classes: {num_classes} -> {class_names}")

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_cnn(image, label):
    image = tf.image.resize(image, [128, 128])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def preprocess_vgg16(image, label):
    image = tf.image.resize(image, [64, 64])
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.one_hot(label, depth=num_classes)

def preprocess_resnet50(image, label):
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.resnet50.preprocess_input(image)
    return image, tf.one_hot(label, depth=num_classes)

# Pipeline caching and prefetching
train_cnn = ds_train_raw.map(preprocess_cnn, AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_cnn   = ds_val_raw.map(preprocess_cnn, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_cnn  = ds_test_raw.map(preprocess_cnn, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_vgg = ds_train_raw.map(preprocess_vgg16, AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_vgg   = ds_val_raw.map(preprocess_vgg16, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_vgg  = ds_test_raw.map(preprocess_vgg16, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_res = ds_train_raw.map(preprocess_resnet50, AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_res   = ds_val_raw.map(preprocess_resnet50, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_res  = ds_test_raw.map(preprocess_resnet50, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:
# Step 4: Model 1 — Custom Deep CNN with Swish Activation
def swish(x):
    return x * tf.nn.sigmoid(x)

def build_cnn_scratch():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.Conv2D(32, (3, 3), padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.Activation(swish),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.Activation(swish),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.Activation(swish),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.Activation(swish),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.Activation(swish),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.Activation(swish),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.Activation(swish),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

cnn_model = build_cnn_scratch()
cnn_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_cnn = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ModelCheckpoint('CNN_best_model.h5', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

print("Training Custom CNN (12 Epochs)... ")
history_cnn = cnn_model.fit(train_cnn, validation_data=val_cnn, epochs=12, callbacks=callbacks_cnn)

In [ ]:
# Step 5: Model 2 — Transfer Learning with VGG16
def build_vgg16():
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(64, 64, 3))
    base_model.trainable = False
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs=base_model.input, outputs=outputs)
    return model

vgg_model = build_vgg16()
vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_vgg = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ModelCheckpoint('vgg16_eurosat_tfds_finetuned.h5', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

print("Training VGG16 Baseline (12 Epochs)... ")
history_vgg = vgg_model.fit(train_vgg, validation_data=val_vgg, epochs=12, callbacks=callbacks_vgg)

In [ ]:
# Step 6: Model 3 — Transfer Learning with ResNet50
def build_resnet50():
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs=base_model.input, outputs=outputs)
    return model

resnet_model = build_resnet50()
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_resnet = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ModelCheckpoint('ResNet_best_model.h5', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

print("Training ResNet50 Baseline (12 Epochs)... ")
history_resnet = resnet_model.fit(train_res, validation_data=val_res, epochs=12, callbacks=callbacks_resnet)

In [ ]:
# Step 7: Benchmark Evaluation & Comparative Metrics Matrix
def evaluate_model(model, test_dataset, is_sparse=False):
    y_true, y_pred, latencies = [], [], []
    
    for batch_x, batch_y in test_dataset:
        start_time = time.time()
        preds = model(batch_x, training=False)
        latency = (time.time() - start_time) * 1000 / len(batch_x)
        latencies.append(latency)
        
        pred_labels = np.argmax(preds.numpy(), axis=1)
        if is_sparse:
            true_labels = batch_y.numpy()
        else:
            true_labels = np.argmax(batch_y.numpy(), axis=1)
            
        y_true.extend(true_labels)
        y_pred.extend(pred_labels)
        
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    
    return {
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(prec * 100, 2),
        "Recall (%)": round(rec * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2),
        "Latency (ms/sample)": round(float(np.mean(latencies)), 2)
    }

print("Running comparative evaluations on test split...")
results = {
    "Custom CNN (Swish)": evaluate_model(cnn_model, test_cnn, is_sparse=True),
    "VGG16 Baseline": evaluate_model(vgg_model, test_vgg, is_sparse=False),
    "ResNet50 Transfer": evaluate_model(resnet_model, test_res, is_sparse=False)
}

df_benchmark = pd.DataFrame(results).T
print("\n" + "="*70)
print("               EUROSAT RGB BENCHMARK COMPARISON MATRIX")
print("="*70)
display(df_benchmark)

In [ ]:
# Step 8: Automatic Hugging Face Model Sync
if HF_TOKEN:
    print(f"Syncing weights to Hugging Face: {HF_REPO_ID}...")
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    
    artifacts = [
        "CNN_best_model.h5",
        "vgg16_eurosat_tfds_finetuned.h5",
        "ResNet_best_model.h5"
    ]
    
    for fname in artifacts:
        if os.path.exists(fname):
            print(f"Uploading {fname} ({os.path.getsize(fname):,} bytes)...")
            api.upload_file(
                path_or_fileobj=fname,
                path_in_repo=fname,
                repo_id=HF_REPO_ID,
                repo_type="model"
            )
            print(f"Uploaded {fname} successfully.")
        else:
            print(f"Warning: {fname} was not found on disk.")
            
    print("\nAll benchmark models successfully synced to Hugging Face!")
    print(f"Repository URL: https://huggingface.co/{HF_REPO_ID}")
else:
    print("Hugging Face token not provided; saved weights remain local.")